# MCD-rPPG ② 复现训练与测试（用仓库原版代码）

**前提**：已经跑过 `mcd_rppg_1_build_dataset_drive.ipynb`，Drive 上有 `mcd_rppg_ds/`。

两条路，按你①里选了哪种产物走：

| 路线 | 需要的产物 | 训练入口 | 说明 |
|---|---|---|---|
| **A. 官方 drop-in** | `preprocessed/mcd_rppg__<pid>__<step>.npz`（含 `video/landmarks/ppg`） | `rppglib.train.train_fold` + `train_SCNN_8roi_mcd_rppg.ipynb` 原样 | 最忠实复现；`rPPGDataset` + `SCNN_8rois.video_processing` 全部跑作者的代码 |
| **B. roi 特征** | `roi/<pid>_<step>_<view>.npz`（含 `roi_rgb/roi_pos/ppg`） | 自写 30 行 Dataset + **同一个** `SCNN_8rois.train/predict` | 数据只有 1.5 GB，特征已经算好，训练快 5–10 倍 |

两条路训练出来的模型**架构、损失、优化器完全相同**（都是 `SCNN_8roi_NN` + Adam(1e-3) + MSE，50 epoch），区别只在"32 维特征是在 Dataset 里现算，还是提前算好存下来"。

下面依次：环境 → 路径补丁 → 路线A → 路线B → 评测 → 复现踩坑记录。

## 1. 环境 + 克隆仓库
仓库 `rppglib` 的依赖只有：`face_alignment`（被 `face_utils` import，训练时其实不用）、`skvideo`（被 `data_utils` import）、`torch/torchaudio`、`cv2`、`numba`、`matplotlib`。Colab 自带大部分。

In [ ]:
%%bash
cd /content
if [ ! -d mcd_rppg ]; then git clone -q --depth 1 https://github.com/ksyegorov/mcd_rppg.git; fi
pip -q install scikit-video "face-alignment==1.4.1" matplotlib
cd /content/mcd_rppg
python - <<'EOF'
import sys; sys.path.insert(0, ".")
import rppglib.train, rppglib.models, torch
print("repo ok | torch", torch.__version__, "| cuda", torch.cuda.is_available())
import rppglib.models.SCNN_8rois as m
print("model ok | 参数量", sum(p.numel() for p in m.SCNN_8roi_NN().parameters())/1e6, "M")
EOF

## 2. 挂盘 + 定位数据集

In [ ]:
import os, sys, glob
import numpy as np, pandas as pd, torch

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DS_ROOT = "/content/drive/MyDrive/mcd_rppg_ds"
REPO    = "/content/mcd_rppg"
sys.path.insert(0, REPO)
os.chdir(REPO)

print("roi npz      :", len(glob.glob(f"{DS_ROOT}/roi/*.npz")))
print("official npz :", len(glob.glob(f"{DS_ROOT}/preprocessed/*.npz")))
print("faces mp4    :", len(glob.glob(f"{DS_ROOT}/faces/*.mp4")))
idx = pd.read_parquet(f"{DS_ROOT}/index.parquet") if os.path.exists(f"{DS_ROOT}/index.parquet") else None
print("index        :", None if idx is None else idx.shape)

## 3. 路径补丁（这三行搞定官方代码的硬编码路径）
- `rppglib/params.py` 里 `/gim/lv03/datasets/rppg/preprocessed` → 你的 Drive
- `rppglib/train.py` 里 `pd.read_csv(f'{config.train_dataset}.csv')` → 读 cwd，所以把改写后的 csv 放回仓库目录
- `SCNN_8rois.video_processing` 会往 `cache/` 写特征缓存，**目录必须先建**，否则第一轮就崩

In [ ]:
import shutil
import rppglib.params as P

P.PREPROCESSED_PATH = DS_ROOT
for k in list(P.DATASET_PATHS):
    P.DATASET_PATHS[k] = DS_ROOT

os.makedirs("results", exist_ok=True)
os.makedirs("cache", exist_ok=True)      # ★ 不建必崩
if os.path.exists("mcd_rppg.csv") and not os.path.exists("mcd_rppg_orig.csv"):
    shutil.copy("mcd_rppg.csv", "mcd_rppg_orig.csv")
print("patched:", P.PREPROCESSED_PATH, "| cwd:", os.getcwd())

## 4. 路线 A：官方 drop-in —— 生成 `mcd_rppg.csv` 并调 `train_fold`

`rPPGDataset` 只用到 csv 的 `file` 和 `fold` 两列，所以只要把 `file` 指到你 Drive 上的 npz 即可。fold 沿用官方按病人的划分（同一人的 before/after 永远同折）。

In [ ]:
from pathlib import Path

rows = []
for p in sorted(Path(DS_ROOT, "preprocessed").glob("mcd_rppg__*.npz")):
    stem = p.stem                                   # mcd_rppg__1020__after
    pid = int(stem.split("__")[1])
    step = stem.split("__")[2]
    rows.append(dict(file=str(p), patient_id=pid, step=step[:6] if step.startswith("before") else step,
                     view="front" if step in ("before", "after") else step.split("_", 1)[1]))

orig = pd.read_csv("mcd_rppg_orig.csv")
orig["pid"] = orig["file"].str.extract(r"mcd_rppg__(\d+)__")[0].astype(int)
orig["stp"] = orig["file"].str.extract(r"mcd_rppg__\d+__(\w+)\.npz")[0]
fold_map = {(int(a), b): int(c) for a, b, c in zip(orig.pid, orig.stp, orig.fold)}

df = pd.DataFrame(rows)
if len(df):
    df["fold"] = [fold_map.get((int(a), b), int(a) % 5) for a, b in zip(df.patient_id, df.step)]
    df.to_csv("mcd_rppg.csv", index=False)
    print("写入 mcd_rppg.csv:", len(df), "条 | fold 分布", df.fold.value_counts().sort_index().to_dict())
    print(df.head(3).to_string(index=False))
else:
    print("⚠️ Drive 上没有 preprocessed/*.npz —— 回 ① 把 SAVE_OFFICIAL=True 再跑一遍（并设 MAX_FRAMES=600）")

### 4.1 冒烟：先确认 `rPPGDataset` 能吃你的 npz
`SCNN_8rois.video_processing` 会对每段做 8 个 ROI 的 POS（首次较慢，之后走 `cache/` 缓存）。先跑 5 条看看。

In [ ]:
import rppglib.train as T
from rppglib.models.SCNN_8rois import SCNN_8rois

class config:                       # 与官方 train_SCNN_8roi_mcd_rppg.ipynb 同名同义
    fps = 30
    ppg_low_freq = 0.5
    ppg_high_freq = 10
    batch_size = 32
    num_workers = 2
    train_dataset = "mcd_rppg"
    test_datasets = ["mcd_rppg"]
    num_folds = 5
    test_fold = 0
    valid_fold = 1
    model = "SCNN_8rois"
    results_folder = "results"
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

df = pd.read_csv("mcd_rppg.csv")
model = SCNN_8rois(config)
ds = T.rPPG_Dataset(df["file"].values[:5], model.video_processing, config)
x, y = ds[0]
print("单条样本:", "features", np.asarray(x).shape, "| ppg", np.asarray(y).shape)
print("送进网络:", tuple(model.net(torch.from_numpy(np.asarray(x, np.float32))[None]).shape))

### 4.2 训练 4 折 + 在 fold 0 上测试
和官方 notebook 完全一致：`test_fold=0`，`valid_fold=1..4`，即 4 次"3 折训 / 1 折验 / fold0 测"。
每折 50 epoch，权重存 `results/SCNN_8rois__mcd_rppg__<fold>.pt`（只保留最后一轮，见后面"踩坑"）。

In [ ]:
import time
res_a = []
for vf in range(1, config.num_folds):
    config.valid_fold = vf
    t0 = time.time()
    cfg = T.train_fold(config)                     # ← 作者原版函数
    res_a.append(dict(valid_fold=vf,
                      ppg=cfg.test_results["test__mcd_rppg__ppg"],
                      hr=cfg.test_results["test__mcd_rppg__hr"],
                      min=round((time.time() - t0) / 60, 1)))
    print(f"fold {vf} done | PPG MAE {res_a[-1]['ppg']:.3f} | HR MAE {res_a[-1]['hr']:.2f} "
          f"| {res_a[-1]['min']} min")
pd.DataFrame(res_a)

## 5. 路线 B：roi 特征 —— 复用同一个 `SCNN_8rois` 类，数据只 1.5 GB

**关键顺序**：`SCNN_8roi_NN` 的第一层卷积核是 `(4,5)`，作用在 32 维通道上 → 32 维必须是 **8 个 ROI × [R, G, B, POS]** 的顺序，不能是 `[24 个 RGB 拼在前, 8 个 POS 拼在后]`。

`SCNN_8rois.train()` / `.predict()` / `SCNN_8roi_NN` 全部沿用作者的代码，我只替换"数据怎么来"。

In [ ]:
from torch.utils.data import Dataset, DataLoader
from rppglib.models.SCNN_8rois import SCNN_8rois
import rppglib.train as T

WINDOW = 600          # 官方 npz 就是 600 帧（20 s @30fps）


class RoiDS(Dataset):
    def __init__(self, keys, root, train=False):
        self.keys, self.root, self.train = list(keys), root, train

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        z = np.load(f"{self.root}/roi/{self.keys[i]}.npz")
        rgb, pos, ppg = z["roi_rgb"], z["roi_pos"], z["ppg"]
        # (T,8,3)+(T,8) → (T,8,4) → (T,32)，顺序 = 每个 ROI 的 [R,G,B,POS]
        x = np.stack([rgb[..., 0], rgb[..., 1], rgb[..., 2], pos], -1).reshape(len(rgb), 32)
        n = min(len(x), len(ppg))
        x, ppg = x[:n], ppg[:n]
        if n > WINDOW:
            s = np.random.randint(n - WINDOW) if self.train else 0
            x, ppg = x[s:s + WINDOW], ppg[s:s + WINDOW]
        ppg = ppg - ppg.mean()
        ppg = ppg / (ppg.std() + 1e-9)
        return torch.from_numpy(x.astype(np.float32)), torch.from_numpy(ppg.astype(np.float32))


keys_all = pd.Series([os.path.basename(p)[:-4] for p in glob.glob(f"{DS_ROOT}/roi/*.npz")])
fold_of = {}
if idx is not None:
    fold_of = dict(zip(idx.key, idx.fold))
folds = np.array([fold_of.get(k, int(k.split("_")[0]) % 5) for k in keys_all])
print("可用记录:", len(keys_all), "| fold 分布", pd.Series(folds).value_counts().sort_index().to_dict())

ds0 = RoiDS(keys_all[:4], DS_ROOT)
xb, yb = next(iter(DataLoader(ds0, batch_size=4)))
print("batch:", tuple(xb.shape), tuple(yb.shape), "→ 网络输出", tuple(SCNN_8rois(config).net(xb).shape))

In [ ]:
def run_roi_fold(vf, test_fold=0, bs=32, workers=2, epochs_note=""):
    tr = keys_all[(folds != test_fold) & (folds != vf)]
    va = keys_all[folds == vf]
    te = keys_all[folds == test_fold]
    dl_tr = DataLoader(RoiDS(tr, DS_ROOT, True), batch_size=bs, num_workers=workers, shuffle=True)
    dl_va = DataLoader(RoiDS(va, DS_ROOT, False), batch_size=bs, num_workers=workers, shuffle=False)
    dl_te = DataLoader(RoiDS(te, DS_ROOT, False), batch_size=bs, num_workers=workers, shuffle=False)

    cfg = type("cfg", (), dict(model="SCNN_8rois", train_dataset="mcd_rppg_roi", valid_fold=vf,
                               results_folder="results", device=config.device, fps=30,
                               ppg_low_freq=0.5, ppg_high_freq=10))()
    m = SCNN_8rois(cfg)                 # 作者的类
    m.train(dl_tr, dl_va)               # 作者的训练循环（50 epoch, Adam 1e-3, MSE）
    trues, preds = [], []
    m.net.eval()
    with torch.no_grad():
        for x, y in dl_te:
            preds.append(m.predict(x).numpy())
            trues.append(y.numpy())
    true = np.concatenate(trues)
    pred = np.concatenate(preds)
    return T.calc_metrics(true, pred), len(tr), len(va), len(te)


res_b = []
for vf in range(1, 5):
    t0 = time.time()
    (ppg_mae, hr_mae), ntr, nva, nte = run_roi_fold(vf)
    res_b.append(dict(valid_fold=vf, ppg=ppg_mae, hr=hr_mae,
                      n_train=ntr, n_test=nte, min=round((time.time() - t0) / 60, 1)))
    print(f"fold {vf} | PPG MAE {ppg_mae:.3f} | HR MAE {hr_mae:.2f} | {res_b[-1]['min']} min")
pd.DataFrame(res_b)

## 6. 结果汇总 + 和论文对照

论文表 3 里本模型在 MCD-rPPG 上：**PPG MAE 0.68 ± 0.03 / HR MAE 4.86 ± 0.36**（5 折平均）。
注意论文的数字是"train/valid/test 全用 600 帧、帧率 30"的前提；侧脸机位是 24 fps，直接套 `fs=30` 估心率会系统性偏高 25%（见第 8 节），所以侧脸 HR 数字不要直接对比论文。

In [ ]:
import numpy as np, pandas as pd

def show(name, res):
    if not res:
        print(f"{name}: 没跑")
        return
    d = pd.DataFrame(res)
    print(f"{name}: PPG MAE {d.ppg.mean():.3f} ± {d.ppg.std():.3f} | "
          f"HR MAE {d.hr.mean():.2f} ± {d.hr.std():.2f}  (n={len(d)} folds)")
    print(d.to_string(index=False))


show("路线A 官方drop-in", globals().get("res_a", []))
show("路线B roi特征   ", globals().get("res_b", []))
print("\n论文参考: PPG MAE 0.68 ± 0.03 | HR MAE 4.86 ± 0.36")

## 7. 评测：按机位拆开 + 用真实 fps 算心率

这是这个数据集真正的卖点（论文表 4）：**正脸 0.68 / 侧脸 1.10（PPG MAE），侧脸 HR 会掉到 14**。
`rppglib.train.calc_metrics` 里 `calculate_fft_hr` 的 `fs` 写死 30 —— 侧脸是 24 fps，所以我自己写一份按记录 fps 的版本。

In [ ]:
import rppglib.processing as PR


def metrics_with_true_fps(trues, preds, fpss, lo=0.5, hi=3.5):
    # PPG MAE + HR MAE，每段用自己的 fps（避免 24fps 被当成 30fps）
    ppg_mae = np.abs(preds - trues).mean(axis=1).mean()
    hr_t, hr_p = [], []
    for i in range(len(trues)):
        fs = float(fpss[i])
        hr_t.append(PR.calculate_fft_hr(trues[i], fs=fs, low_pass=lo, high_pass=hi))
        hr_p.append(PR.calculate_fft_hr(preds[i], fs=fs, low_pass=lo, high_pass=hi))
    hr_mae = np.abs(np.array(hr_t) - np.array(hr_p)).mean()
    return float(ppg_mae), float(hr_mae), float(np.mean(hr_t)), float(np.mean(hr_p))


def eval_by_view(vf=1, ckpt=None):
    ck = ckpt or f"results/SCNN_8rois__mcd_rppg_roi__{vf}.pt"
    if not os.path.exists(ck):
        print("没有权重:", ck, "→ 先跑第 5 节")
        return
    te = keys_all[folds == 0]
    if len(te) == 0:
        print("测试折为空：检查 index.parquet 的 fold 列")
        return
    m = SCNN_8rois(type("cfg", (), dict(model="SCNN_8rois", train_dataset="mcd_rppg_roi",
                                        valid_fold=vf, results_folder="results",
                                        device=config.device))())
    m.net.load_state_dict(torch.load(ck, map_location="cpu"))
    m.net.cpu().eval()
    rows = []
    with torch.no_grad():
        for k in te:
            x, y = RoiDS([k], DS_ROOT, False)[0]
            p = m.net(x[None])[0].numpy()
            fs = float(np.load(f"{DS_ROOT}/roi/{k}.npz")["fps"])
            pmae, hmae, hrt, hrp = metrics_with_true_fps(y.numpy()[None], p[None], [fs])
            rows.append(dict(key=k, view=k.split("_")[-1], fps=fs, ppg=pmae,
                             hr_mae=hmae, hr_true=hrt, hr_pred=hrp))
    d = pd.DataFrame(rows)
    print(d.groupby("view")[["ppg", "hr_mae", "hr_true", "hr_pred"]].mean().round(2))
    return d


d_view = eval_by_view(vf=1)

## 8. 复现踩坑记录（我在源码里核对过的，按重要性排序）

1. **`SCNN_8rois.train()` 的 "best" 存盘是坏的**：
   ```python
   for i in range(NUM_EPCHS):
       ...
       best_valid_loss = 1e10          # ← 每轮都被重置
       if valid_loss < best_valid_loss: torch.save(...)
   ```
   `best_valid_loss` 写在了 epoch 循环**内部**，所以每一轮都会存盘 → 最终权重 = **最后一轮**，不是最优轮。想按 best 存得自己改（把 `best_valid_loss` 提到循环外，并 `min` 而不是累加 `valid_loss`——另外 `valid_loss` 是整个 epoch 的**累加和**，不是均值，所以轮与轮之间不可比）。
2. **`torchPOS` 依赖 `torchaudio.functional.filtfilt`**。Colab 上 torch/torchaudio 版本必须配套，否则 `import rppglib.models` 直接失败。装不上就 `pip install torchaudio --index-url https://download.pytorch.org/whl/cu121` 对齐。
3. **`cache/` 目录必须先建**：`SCNN_8rois.video_processing` 里 `np.save(f'cache/{hash}.npy')` 不会自建目录。
4. **`calc_metrics` 的 `fs` 写死 30**：侧脸（IriunWebcam 24 fps）算 HR 会偏高，见第 7 节的自写版本。
5. **`rPPGDataset` 一次性把所有视频读进内存**（`self.videos = [open_and_preprocess_video(f) for f in ...]`）。600 帧 96×96 尚可，全时长 4800 帧会爆内存 → 路线 A 必须用 600 帧的 npz。
6. **`mcd_rppg.csv` 覆盖范围只有正脸 1200 条**；侧脸要自己加行（本 notebook 第 4 节已经按 `_left/_right` 后缀处理）。
7. **`train.py` 里 test 只用 `config.test_fold`**，所以 4 次训练都是在 fold0 上测，fold0 成了固定的 held-out —— 与论文"5 折平均"略有差别（论文是把 5 折轮流当 test）。想完全对齐论文就再把 `test_fold` 轮转一遍，代价是 5×训练时间。
8. **batch 内 T 必须一致**：官方 npz 全是 600 帧，所以能 `torch.stack`；自己造全时长 npz 时要么定长切窗（本 notebook 路线 B 的 `WINDOW=600` 随机裁剪就是这么干的），要么 batch_size=1。